# KM V8 events and AFT covariates — single raw pull

For each market-day this workflow pulls the preceding, target, and following raw trade days once; constructs one event-time stream; detects regular-grid 1s Lee–Mykland shocks for both spot-first and perp-first perspectives; applies the saved event-store V5 resolution rules; and attaches leakage-safe base covariates from the same in-memory data. Daily checkpoints make the run resumable and memory is released after every market-day.

In [10]:
from pathlib import Path
import gc
import os
import pandas as pd

from km_v8_regular_lm_pilot import (
    MARKETS, METHODOLOGY_VERSION, atomic_csv,
    plot_btc_km_month_comparison, run_date_block,
)
from survival_analysis_data_processing_final import augment_liquidity_metrics
from survival_analysis_data_pull_final import pull_or_load_market_metrics
from survival_analysis_utils_final import (
    COVARIATE_TIMING_VERSION, parquet_has_timing_version, validate_augmented_timing,
    validate_base_covariate_timing,
)

In [11]:
# Full AFT pull; END_DATE is exclusive.
START_DATE = os.getenv('KMV8_START_DATE', '2021-01-01')
END_DATE = os.getenv('KMV8_END_DATE', '2026-01-01')
KM_PLOT_MONTHS = ('2021-12', '2022-12', '2023-12', '2025-12')
MARKET_NAMES = os.getenv('KMV8_MARKETS', 'btc_um,btc_cm,eth_um,eth_cm').split(',')
GRID = '1s'
SIGNIFICANCE = 0.01
OUTPUT_ROOT = Path(os.getenv('KMV8_OUTPUT_ROOT', 'sa_results/km_v8_final_01'))
BATCH_DAYS = 17
RUN_LIQUIDITY_AUGMENTATION = os.getenv('KMV8_LIQUIDITY', '1') == '1'
REDOWNLOAD_METRICS = False

DATES = pd.date_range(
    START_DATE, END_DATE, inclusive='left', freq='D'
).strftime('%Y-%m-%d').tolist()
DATE_BATCHES = [
    DATES[offset:offset + BATCH_DAYS]
    for offset in range(0, len(DATES), BATCH_DAYS)
]
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(METHODOLOGY_VERSION, COVARIATE_TIMING_VERSION, len(DATES), 'days', 'batch=', BATCH_DAYS)

v8_regular_lm_daily_v4 prior_day_completed_ffill_1s_event_tick_minus_1_5min_final_v4 1826 days batch= 17


In [ ]:
# One raw pull per 17 target days; LM/event logic remains independent by UTC day.
summary_path = OUTPUT_ROOT / 'comparison_summary.csv'
summary = pd.read_csv(summary_path) if summary_path.exists() else pd.DataFrame()

def day_complete(market, date):
    invalid_path = OUTPUT_ROOT / 'invalid_days.csv'
    if invalid_path.exists():
        invalid = pd.read_csv(invalid_path)
        marked = invalid[
            invalid['market'].eq(market)
            & invalid['date'].astype(str).eq(date)
            & invalid['methodology_version'].eq(METHODOLOGY_VERSION)
        ]
        if not marked.empty:
            return True
    if summary.empty or not {'methodology_version', 'aft_file'}.issubset(summary.columns):
        return False
    prior = summary[
        summary['market'].eq(market)
        & summary['date'].astype(str).eq(date)
        & summary['grid'].eq(GRID)
        & summary['methodology_version'].eq(METHODOLOGY_VERSION)
    ]
    return (prior['first'].nunique() == 2
            and prior['event_file'].map(lambda p: Path(p).exists()).all()
            and prior['aft_file'].map(lambda p: parquet_has_timing_version(p)).all())

for market in MARKET_NAMES:
    for batch in DATE_BATCHES:
        if all(day_complete(market, date) for date in batch):
            print('[skip verified block]', market, batch[0], batch[-1])
            continue
        rows = run_date_block(
            market=market, dates=batch, grids=[GRID],
            output_root=OUTPUT_ROOT, significance=SIGNIFICANCE,
            build_covariates=True,
        )
        summary = pd.concat([summary, pd.DataFrame(rows)], ignore_index=True)
        summary = summary.drop_duplicates(['market', 'date', 'first', 'grid'], keep='last')
        atomic_csv(summary.sort_values(['market', 'date', 'grid', 'first']), summary_path)
        gc.collect()

print('completed rows:', len(summary))

[load block] btc_um 2021-01-01..2021-01-17 (19 archive days)
Processing 19 files...
Processing batch 1/1
Combining results...
Processing 19 files...
Processing batch 1/1
Combining results...
  [grid] 1s
  [grid] 1s
  [grid] 1s
  [grid] 1s
  [grid] 1s
  [grid] 1s
  [grid] 1s
  [grid] 1s
  [grid] 1s
  [grid] 1s
  [grid] 1s
  [grid] 1s
  [grid] 1s
  [grid] 1s
  [grid] 1s
  [grid] 1s
  [grid] 1s
[load block] btc_um 2021-01-18..2021-02-03 (19 archive days)
Processing 19 files...
Processing batch 1/1
Combining results...
Processing 19 files...
Processing batch 1/1
Combining results...
  [grid] 1s
  [grid] 1s
  [grid] 1s
  [grid] 1s
  [grid] 1s
  [grid] 1s
  [grid] 1s
  [grid] 1s
  [grid] 1s
  [grid] 1s
  [grid] 1s
  [grid] 1s
  [grid] 1s
  [grid] 1s
  [grid] 1s
  [grid] 1s
  [grid] 1s
[load block] btc_um 2021-02-04..2021-02-20 (19 archive days)
Processing 19 files...
Processing batch 1/1
Combining results...
Processing 19 files...
Processing batch 1/1
Combining results...
  [grid] 1s
  [grid

PanicException: no read method found: PyErr { type: <class 'MemoryError'>, value: MemoryError('Unable to allocate output buffer.'), traceback: Some("Traceback (most recent call last):\n  File \"c:\\Users\\Julia\\anaconda3\\Lib\\zipfile\\__init__.py\", line 1001, in read\n    buf += self._read1(self.MAX_N)\n  File \"c:\\Users\\Julia\\anaconda3\\Lib\\zipfile\\__init__.py\", line 1091, in _read1\n    data = self._decompressor.decompress(data, n)\n") }

: 

In [ ]:
# Pool daily base-covariate checkpoints into monthly files used downstream.
monthly_root = OUTPUT_ROOT / 'aft_data_monthly'
for market in MARKET_NAMES:
    for first in ('spot', 'perp'):
        paths = [
            OUTPUT_ROOT / 'aft_data' / market / f'{first}_{date}_{GRID}.parquet'
            for date in DATES
            if (OUTPUT_ROOT / 'aft_data' / market / f'{first}_{date}_{GRID}.parquet').exists()
        ]
        frames = [pd.read_parquet(path) for path in paths]
        if not frames:
            continue
        pooled = pd.concat(frames, ignore_index=True).drop_duplicates(['start_ts', 'resolution_pct'])
        validate_base_covariate_timing(pooled, f'{market} {first} pooled')
        for period, month in pooled.groupby(pd.to_datetime(pooled['start_ts']).dt.to_period('M')):
            out = monthly_root / market / f'{first}_{period}.parquet'
            out.parent.mkdir(parents=True, exist_ok=True)
            month.sort_values('start_ts').to_parquet(out, index=False, compression='zstd')
        del frames, pooled
        gc.collect()

In [ ]:
# Optional: attach open interest and long/short metrics strictly before each event.
if RUN_LIQUIDITY_AUGMENTATION:
    for market in MARKET_NAMES:
        symbol, cm_um, _ = MARKETS[market]
        inputs = [
            path for path in sorted((monthly_root / market).glob('*.parquet'))
            if path.stem[-7:] in {date[:7] for date in DATES}
        ]
        metrics = pull_or_load_market_metrics(
            OUTPUT_ROOT / 'metric_runs' / market, symbol, cm_um, inputs,
            redownload=REDOWNLOAD_METRICS,
        )
        for path in inputs:
            out = OUTPUT_ROOT / 'aft_data_liquidity_monthly' / market / path.name
            out.parent.mkdir(parents=True, exist_ok=True)
            augmented = augment_liquidity_metrics(pd.read_parquet(path), metrics)
            validate_augmented_timing(augmented, path.name)
            augmented.to_parquet(out, index=False, compression='zstd')
            del augmented
        del metrics
        gc.collect()

Reading metrics cache: sa_results\km_v8_final_single_pull\metric_runs\btc_um\metrics_cache\BTCUSDT_um_liquidity_metrics.csv.gz
Reading metrics cache: sa_results\km_v8_final_single_pull\metric_runs\btc_cm\metrics_cache\BTCUSD_PERP_cm_liquidity_metrics.csv.gz
Reading metrics cache: sa_results\km_v8_final_single_pull\metric_runs\eth_um\metrics_cache\ETHUSDT_um_liquidity_metrics.csv.gz
Reading metrics cache: sa_results\km_v8_final_single_pull\metric_runs\eth_cm\metrics_cache\ETHUSD_PERP_cm_liquidity_metrics.csv.gz


In [ ]:
# KM comparison graph and final checkpoint audit.
summary = pd.read_csv(summary_path)
current = summary[
    summary['methodology_version'].eq(METHODOLOGY_VERSION)
    & summary['market'].isin(MARKET_NAMES)
    & summary['date'].astype(str).isin(DATES)
    & summary['grid'].eq(GRID)
    & summary['first'].isin(['spot', 'perp'])
].copy()
invalid_path = OUTPUT_ROOT / 'invalid_days.csv'
invalid = pd.read_csv(invalid_path) if invalid_path.exists() else pd.DataFrame()
invalid_pairs = set()
if not invalid.empty:
    invalid = invalid[
        invalid['methodology_version'].eq(METHODOLOGY_VERSION)
        & invalid['market'].isin(MARKET_NAMES)
        & invalid['date'].astype(str).isin(DATES)
    ]
    invalid_pairs = set(zip(invalid['market'], invalid['date'].astype(str)))
    current = current[
        ~current.apply(lambda r: (r['market'], str(r['date'])) in invalid_pairs, axis=1)
    ].copy()
key = ['market', 'date', 'first', 'grid']
assert not current.duplicated(key).any(), current.loc[current.duplicated(key, keep=False), key]
expected_keys = pd.MultiIndex.from_product(
    [MARKET_NAMES, DATES, ['spot', 'perp'], [GRID]], names=key
)
if invalid_pairs:
    invalid_keys = pd.MultiIndex.from_tuples(
        [(market, date, first, GRID)
         for market, date in invalid_pairs for first in ('spot', 'perp')],
        names=key,
    )
    expected_keys = expected_keys.difference(invalid_keys)
actual_keys = pd.MultiIndex.from_frame(current[key])
missing = expected_keys.difference(actual_keys)
unexpected = actual_keys.difference(expected_keys)
assert missing.empty and unexpected.empty, {
    'actual': len(actual_keys), 'expected': len(expected_keys),
    'missing': missing[:10].tolist(), 'unexpected': unexpected[:10].tolist(),
}
km_plot_120s = plot_btc_km_month_comparison(
    OUTPUT_ROOT, months=KM_PLOT_MONTHS, grid=GRID, horizon=120.0,
    plot_step_seconds=0.25, require_complete=False,
)
km_plot_20s = plot_btc_km_month_comparison(
    OUTPUT_ROOT, months=KM_PLOT_MONTHS, grid=GRID, horizon=20.0,
    plot_step_seconds=0.25, require_complete=False,
)
assert current['event_file'].map(lambda p: Path(p).exists()).all()
assert current['aft_file'].map(lambda p: Path(p).exists()).all()
print('verified direction-days:', len(current))
if invalid_pairs:
    print('skipped invalid/error market-days:', len(invalid_pairs))
    print(invalid.sort_values(['market', 'date'])[
        ['market', 'date', 'spot_trade_rows', 'perp_trade_rows', 'reason']
    ].to_string(index=False))
print('KM plot (120s):', km_plot_120s)
print('KM plot (20s):', km_plot_20s)

verified direction-days: 248
KM plot: sa_results\km_v8_final_single_pull\plots\km_curves_1s_v5_vs_v8.png
